In [1]:
# --- Colab installs (single line for speed) ---
# langchain            : High-level orchestration for LLM apps (Chains, Agents, Runnables).
# langchain-core       : Core runtime pieces (PromptTemplate, OutputParsers, Runnable*).
# langchain-community  : Community integrations (tools/utilities like WikipediaAPIWrapper, SQL, etc.).
# langchain-groq       : Groq LLM provider so LangChain can call Groq models.
# wikipedia            : Python Wikipedia client used by the WikipediaAPIWrapper.
!pip -q install langchain langchain-core langchain-community langchain-groq wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import getpass
import os

if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API key for Groq: ")

Enter API key for Groq: ··········


In [3]:
# LLM provider: LangChain wrapper around the Groq Chat API (you'll pass your GROQ_API_KEY)
from langchain_groq import ChatGroq
# Prompt templating: build structured prompts (system/human) and inject variables safely
from langchain_core.prompts import ChatPromptTemplate
# Output parsing: convert model responses to plain strings (or later to structured objects)
from langchain_core.output_parsers import StrOutputParser
# Runnables: compose steps into pipelines
# - RunnableLambda wraps a Python function as a pipeline node
# - RunnablePassthrough forwards the input unchanged (handy when wiring dicts)
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
# Tool decorator: turn a Python function into a standardized "Tool"
# (name, description, input schema) that Agents can discover and call
from langchain_core.tools import tool


Use this when we only need one role (usually "human" or "system") with a variable.

In [4]:
prompt = ChatPromptTemplate.from_template("Explain {topic} in one short paragraph.")
# rendered = prompt.format_messages(topic="embeddings")
# print(rendered)
# 1) Make the LLM
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
# 2) Build the chain
chain = prompt | llm | StrOutputParser()
# 3) Run it
print(chain.invoke({"topic": "embeddings"}))


In machine learning, embeddings are a way to represent complex data, such as text or images, as numerical vectors in a high-dimensional space. These vectors, called embeddings, capture the semantic meaning and relationships between data points, allowing models to perform tasks like similarity search, clustering, and classification more effectively. For example, word embeddings like Word2Vec or GloVe represent words as vectors where similar words (e.g., "dog" and "cat") are closer together in the vector space.


## 1) Single-Step Chain (LLM only; fastest path)

Use this when the problem can be solved in one shot (no tools).
You can “flip on” CoT/ToT just by changing the prompt.

In [5]:
# --- 1) Pick an LLM backend (Groq) ---
# ChatGroq is LangChain’s chat interface for Groq models.
#   model        : the exact Groq model id (e.g., "llama-3.1-8b-instant")
#   temperature  : 0 = more deterministic (good for teaching/evals); higher = more creative
# llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# --- 2) Build a chat-style prompt template ---
# ChatPromptTemplate.from_messages([...]) takes a list of (role, template) pairs.
# Roles can be: "system" (behavior), "human" (user input), "ai" (assistant),
# and you can include placeholders like {topic} that will be filled at runtime.
base_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise, friendly tutor."),
    ("human",  "Explain {topic} in 10-20 words for a beginner.")
])

# --- 3) Compose a Chain with the | (pipe) operator ---
# In LangChain, objects that implement the Runnable interface can be chained:
#   prompt -> llm -> output_parser
# The pipe (|) means: take output of the left node and feed to the right node.
# StrOutputParser() converts the model’s AIMessage to a plain string (resp.content).
single_step_chain = base_prompt | llm | StrOutputParser()
# AIMessage(content="Embeddings are vector representations...")


# --- 4) Run the chain ---
# .invoke({...}) executes the pipeline once.
# You pass a dict with values for any {placeholders} used in the prompt.
# print(single_step_chain.invoke({"topic": "Retrieval-Augmented Generation"}))
print(single_step_chain.invoke({"topic": "Embeddings"}))


Embeddings: Representing words or objects as numerical vectors to capture their semantic meaning and relationships in a high-dimensional space.


In [6]:
inputs = [
    {"topic": "Embeddings"},
    {"topic": "Transformers"}
]

results = single_step_chain.batch(inputs)
print(results)


['Embeddings: Representing words or objects as numerical vectors to capture their semantic meaning and relationships in a high-dimensional space.', 'Transformers are robots that can change shape, often from vehicles to humans, in a popular animated TV series and movies.']


In [7]:
# # CoT flavor (ask the model to show brief steps)
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Think step-by-step; keep the reasoning brief."),
    ("human",  "Solve: {problem}. Show your reasoning, then give the final answer clearly.")
])
cot_chain = cot_prompt | llm | StrOutputParser()
print(cot_chain.invoke({"problem": "Is 17 prime? Justify briefly."}))




To determine if 17 is prime, we need to check if it has any divisors other than 1 and itself.

Step 1: Check if 17 is divisible by any number less than itself.
- 17 is not divisible by 2 (even number).
- 17 is not divisible by 3 (sum of digits is 8, not a multiple of 3).
- 17 is not divisible by 4 (even number).
- 17 is not divisible by 5 (last digit is not 0 or 5).
- 17 is not divisible by 6 (even number).
- 17 is not divisible by 7 (not a multiple of 7).
- 17 is not divisible by 8 (even number).
- 17 is not divisible by 9 (sum of digits is 8, not a multiple of 9).
- 17 is not divisible by 10 (last digit is not 0).
- 17 is not divisible by 11 (not a multiple of 11).
- 17 is not divisible by 12 (even number).
- 17 is not divisible by 13 (not a multiple of 13).
- 17 is not divisible by 14 (even number).
- 17 is not divisible by 15 (last digit is not 0 or 5).
- 17 is not divisible by 16 (even number).
- 17 is not divisible by 17 (itself, but a prime number is only divisible by 1 and itse

In [8]:
# (multiple candidate paths, choose best)
tot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate 3 solution paths, compare them, choose the best, then answer."),
    ("human",  "Task: {task}. Output format: (Paths)->(Comparison)->(Chosen)->(Answer).")
])
tot_chain = tot_prompt | llm | StrOutputParser()
print(tot_chain.invoke({"task": "Plan 3 short strategies to remember new vocabulary."}))

**(Paths)**

1. **Flashcard Method**: Create physical or digital flashcards with the new vocabulary word on one side and its meaning on the other. Review the cards regularly, covering the meaning side to test recall.
2. **Association Method**: Connect new vocabulary words to words or concepts you already know. Create mental or written associations to help remember the new word's meaning.
3. **Storytelling Method**: Create a short story incorporating the new vocabulary words. The more absurd or memorable the story, the better it will stick in your memory.

**(Comparison)**

- **Flashcard Method**: Effective for large amounts of vocabulary, but can be time-consuming to create and review.
- **Association Method**: Works well for words with strong connections to existing knowledge, but may not be effective for abstract or complex vocabulary.
- **Storytelling Method**: Engaging and memorable, but may not be practical for large amounts of vocabulary or words with no clear connections.

**(Ch

### 2) Multi-Step Chain (two LLM calls; still fixed)

When you want a predictable pipeline (e.g., “clarify → answer”).
Each extra LLM step adds a little latency, but everything is predetermined.

In [9]:
# Step A: clarify the user's query
clarify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the user query to be precise and unambiguous."),
    ("human",  "{q}")
])
clarify = clarify_prompt | llm | StrOutputParser()

# Step B: answer clearly using the clarified query
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer clearly in 3 bullet points."),
    ("human",  "{q}")
])
answer = answer_prompt | llm | StrOutputParser()

# Pipe them together: q -> clarify -> answer
multi_step_chain = (
    {"q": RunnablePassthrough()}                                   # pass raw input
    | clarify                                                       # LLM call #1
    | (lambda clarified: {"q": f"Topic: {clarified}"})              # map to next input
    | answer                                                        # LLM call #2
)

print(multi_step_chain.invoke("tell me about RAG and give classroom examples"))


Here are 3 key points about the RAG System:

* **Visual Progress Tracking**: The RAG system provides a clear and visual representation of progress and performance, making it easier to identify areas that require attention and improvement.
* **Encourages Student Ownership**: By using the RAG system, students are encouraged to take ownership of their learning and behavior, as they can see their progress and performance in a clear and concise manner.
* **Facilitates Communication**: The RAG system facilitates communication between teachers and students about progress and performance, helping to create a supportive and motivating learning environment that helps students achieve their full potential.


### 3) A Chain that uses a Tool (predetermined call order)

Here the pipeline always uses the tool (no LLM “deciding”).
We’ll fetch a short Wikipedia summary (tool) and then summarize it (LLM). This remains a Chain because the order is fixed.

In [14]:
# Simple Python function to fetch from Wikipedia (no API key needed)
from langchain_community.utilities import WikipediaAPIWrapper
wiki = WikipediaAPIWrapper(lang="en", top_k_results=1, doc_content_chars_max=800)

def wiki_fetch(topic: str) -> str:
    return wiki.run(topic)

# Wrap the python function as a Runnable
fetch_node = RunnableLambda(lambda topic: wiki_fetch(topic))

# LLM summarizer node
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the given text in 2 crisp bullet points."),
    ("human",  "{text}")
])
summarizer = summarize_prompt | llm | StrOutputParser()
# Fixed chain: (topic) -> wiki_fetch -> summarizer
wiki_chain = (
    {"text": fetch_node}  # takes the input string (topic) and produces text
    | summarizer
)

print(wiki_chain.invoke("FAST Islamabad"))


Here are 2 crisp bullet points summarizing the given text:

• The National University of Computer and Emerging Sciences (NUCES) is a private research university in Pakistan.
• It is commonly known as Foundation for Advancement of Science and Technology (FAST) and has multiple campuses across different cities in Pakistan.


### 4) Agent (dynamic tool use; LLM decides order)

Now we give the LLM a toolbox and let it decide which tool to use and when.
We’ll create two tools: multiply and wiki_search. The agent will pick the order.

In [15]:
from langchain_core.tools import tool
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents import create_agent

from langchain_groq import ChatGroq  # if you haven't created llm yet

# --- LLM ---
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# --- Wikipedia tool backend ---
wiki = WikipediaAPIWrapper(lang="en", top_k_results=1, doc_content_chars_max=800)

# --- Tools ---
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers a and b."""
    return a * b

@tool
def wiki_search(query: str) -> str:
    """Search Wikipedia and return a short summary."""
    return wiki.run(query)

tools = [multiply, wiki_search]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
  "Use tools when needed. Output exactly 2 parts:\n"
  "1) Math result as a number only.\n"
  "2) Exactly give words what user ask ending with 'Source: Wikipedia'.\n"
  "No apologies. No extra phrases. No headings like Page/Summary."

    ),
)

# --- Run ---
q = "What is 12*7, and give 10 words about Pakistan?"
result = agent.invoke({"messages": [{"role": "user", "content": q}]})

# result usually contains a messages list; print the final assistant message content
if isinstance(result, dict) and "messages" in result and result["messages"]:
    print(result["messages"][-1].content)
else:
    print(result)


84
Pakistan, officially the Islamic Republic of Pakistan, is a country in South Asia. Source: Wikipedia


In [16]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# Quick ping — should print "OK"
print(llm.invoke("Reply with: OK").content)

OK


### 5) Decide at runtime: Chain or Agent?

Use a tiny router chain to decide if the query is simple (chain) or needs tools (agent).
(For class demos, this shows how to pick the right path.)

In [17]:
# --- 2) Simple chain (no tools) ---
base_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise, friendly tutor."),
    ("human",  "Explain {topic} in 10-20 words for a beginner.")
])
single_step_chain = base_prompt | llm | StrOutputParser()

# --- 3) Tools ---
wiki = WikipediaAPIWrapper(lang="en", top_k_results=1, doc_content_chars_max=800)

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers a and b."""
    return a * b

@tool
def wiki_search(query: str) -> str:
    """Search Wikipedia and return a short summary."""
    return wiki.run(query)

tools = [multiply, wiki_search]

# IMPORTANT: bind tools to the model (prevents tool_call format issues)
llm_with_tools = llm.bind_tools(tools)

# --- 4) Modern Agent (tool-calling) ---
agent = create_agent(
    model=llm_with_tools,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant.\n"
        "- Use tools when needed.\n"
        "- For multiplication, call multiply(a, b).\n"
        "- For factual lookup, call wiki_search(query).\n"
        "- If the user asks multiple things, answer all parts.\n"
        "- After tool calls, write the final answer in plain English.\n"
        "- Include the math result clearly.\n"
        "- If Wikipedia was used, end with: Source: Wikipedia.\n"
        "- Never return only the source line.\n"
    ),
)

def ask_agent(q: str) -> str:
    """Call the agent (chat-style) and return final assistant text."""
    res = agent.invoke({"messages": [{"role": "user", "content": q}]})
    return res["messages"][-1].content

# --- 5) Router chain ---
router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Decide routing based on query complexity:\n"
     "- Output 'simple' for: definitions, explanations, rewrites, translations\n"
     "- Output 'needs_tools' for: math calculations, current events, specific facts about places/people, data lookups\n"
     "Examples:\n"
     "- 'Explain machine learning' → simple\n"
     "- 'What is 15 * 23?' → needs_tools\n"
     "- 'Facts about Tokyo' → needs_tools\n"
     "Only respond with 'simple' or 'needs_tools'."),
    ("human", "{query}")
])
router_chain = router_prompt | llm | StrOutputParser()

# --- 6) App runner ---
def run_app(user_query: str) -> str:
    route = router_chain.invoke({"query": user_query}).strip().lower()
    route = "needs_tools" if "needs_tools" in route else "simple"
    print("route:", route)

    if route == "needs_tools":
        print(f"Using AGENT for: '{user_query}'")
        return ask_agent(user_query)
    else:
        print(f"Using CHAIN for: '{user_query}'")
        return single_step_chain.invoke({"topic": user_query})

# --- 7) Tests ---
print(run_app("calculate 12*2?"))
print(run_app("Give one fact about Islamabad."))


route: needs_tools
Using AGENT for: 'calculate 12*2?'
The result of 12*2 is 24.
route: needs_tools
Using AGENT for: 'Give one fact about Islamabad.'
Islamabad is the capital city of Pakistan. Source: Wikipedia.
